# 🧪 ADMET Toxicity Prediction with ChemBERTa
## Milestone 2: Multi-Target Classification across All 12 Tox21 Assays

**What's new vs Milestone 1:**
- **Multi-label output:** The head outputs 12 logits — one per target — instead of 2
- **Masked BCE loss:** NaN entries (untested compounds) are excluded from the loss per-sample
- **Per-target class weights:** Each target gets its own positive weight based on its imbalance ratio
- **Multi-task learning:** All 12 targets share the same ChemBERTa encoder — signal from one target helps all others
- **Per-target thresholds:** Optimal decision thresholds are tuned independently for each assay

**Why multi-task learning helps:**  
A compound that damages DNA tends to activate *multiple* stress-response pathways simultaneously.  
Training on all 12 targets at once lets the model learn shared toxic substructures that generalise  
better than 12 separate single-task models.

**Architecture:**
```
SMILES Input
    ↓
ChemBERTa-77M (frozen + LoRA adapters)
    ↓
Classification Head: Linear(600 → 12)
    ↓
12 independent sigmoid probabilities
[NR-AR, NR-AR-LBD, NR-AhR, ..., SR-p53]
```

**Runtime:** GPU recommended (T4 in Colab, or local MPS on Apple Silicon)

## Section 1: Setup

In [ ]:
# Same dependencies as Milestone 1
!pip install -q transformers datasets peft accelerate wandb scikit-learn huggingface_hub python-dotenv rdkit
# torchvision triggers a broken VideoReader import in datasets — remove it
!pip uninstall -y torchvision 2>/dev/null || true
# peft requires torchao >= 0.16.0; Colab ships 0.10.0
!pip install -q --upgrade torchao

In [2]:
import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datetime import datetime

from datasets import load_dataset, Dataset as HFDataset, Sequence, Value, Features
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
import wandb
from huggingface_hub import login

SEED = 42
set_seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/Users/michaelmalloy/.pyenv/versions/3.12.4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.12.0
CUDA available:  False


### Configuration

Key differences from Milestone 1:
- No `TARGET_TASK` — we use all 12 targets simultaneously
- `NUM_TARGETS = 12` drives the model head and loss function
- `metric_for_best_model` is now `mean_roc_auc` (average across all 12 targets)

In [3]:
# ── Model & Data ──────────────────────────────────────────────────────────────
BASE_MODEL   = "DeepChem/ChemBERTa-77M-MTR"
DATASET_NAME = "scikit-fingerprints/MoleculeNet_Tox21"
PROJECT_NAME = "admet-toxicity"
HF_USER      = "mike-malloy"

# All 12 Tox21 targets — fixed order matters for label tensor indexing
TASK_COLUMNS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]
NUM_TARGETS = len(TASK_COLUMNS)   # 12

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.1
TARGET_MODULES  = ["query", "value"]

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS                   = 10
BATCH_SIZE               = 32
LEARNING_RATE            = 2e-4
WEIGHT_DECAY             = 0.01
WARMUP_STEPS             = 50
MAX_SEQ_LENGTH           = 128
LOG_STEPS                = 10
EVAL_STEPS               = 50
SAVE_STEPS               = 50
EARLY_STOPPING_PATIENCE  = 4

# ── Derived ───────────────────────────────────────────────────────────────────
RUN_NAME       = f"chemberta-tox21-multitarget-scaffold-{datetime.now():%Y%m%d_%H%M}"
HUB_MODEL_NAME = f"{HF_USER}/{RUN_NAME}"

print(f"Model:       {BASE_MODEL}")
print(f"Targets:     {NUM_TARGETS}")
print(f"Run name:    {RUN_NAME}")

Model:       DeepChem/ChemBERTa-77M-MTR
Targets:     12
Run name:    chemberta-tox21-multitarget-20260519_0859


### Login to HuggingFace and Weights & Biases

Same as Milestone 1 — reads from `.env` locally or Colab secrets in the Colab UI.

In [4]:
def _get_secret(key: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key)
    except (ImportError, Exception):
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass
        val = os.environ.get(key)
        if not val:
            raise EnvironmentError(
                f"Secret '{key}' not found. "
                f"Add {key}=<value> to a .env file in the project directory."
            )
        return val

hf_token = _get_secret('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

wandb_api_key = _get_secret('WANDB_API_KEY')
os.environ["WANDB_API_KEY"]   = wandb_api_key
os.environ["WANDB_PROJECT"]   = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"]     = "false"
wandb.login()

wandb.init(
    project=PROJECT_NAME,
    name=RUN_NAME,
    config={
        "base_model":     BASE_MODEL,
        "num_targets":    NUM_TARGETS,
        "task":           "multi_label_classification",
        "lora_r":         LORA_R,
        "lora_alpha":     LORA_ALPHA,
        "lora_dropout":   LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "epochs":         EPOCHS,
        "batch_size":     BATCH_SIZE,
        "learning_rate":  LEARNING_RATE,
        "seed":           SEED,
    }
)
print("\u2705 Logged in to HuggingFace and Weights & Biases")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mike-malloy-2004 (mike-malloy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Logged in to HuggingFace and Weights & Biases


---
## Section 2: Data

### Multi-label vs multi-class

In Milestone 1 the label was a single integer (0 or 1) — a **multi-class** problem.  
Here each compound gets a **vector of 12 floats** — a **multi-label** problem:

```
compound X → [0.0, 0.0, 1.0, None, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
               NR-AR       NR-AhR↑       ...    SR-ARE↑        ...    SR-p53
```

`None` (NaN) means the compound was never tested against that target.  
We use **−1 as a sentinel** for NaN and **mask those entries out of the loss** — they contribute  
no gradient, neither positive nor negative.

In [5]:
print("Loading Tox21 dataset...")
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)

sample = raw_dataset['train'][0]
SMILES_COL = 'smiles' if 'smiles' in sample else 'SMILES'
print(f"\nSMILES column: '{SMILES_COL}'")

Loading Tox21 dataset...
DatasetDict({
    train: Dataset({
        features: ['SMILES', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'],
        num_rows: 7831
    })
})

SMILES column: 'SMILES'


In [6]:
# ── Label distribution across all 12 targets ──────────────────────────────────
df_raw = raw_dataset['train'].to_pandas()

print(f"{'Target':<20} {'N tested':<10} {'N toxic':<10} {'% toxic':<10} {'pos_weight'}")
print("-" * 62)
for t in TASK_COLUMNS:
    col      = df_raw[t]
    n_tested = col.notna().sum()
    n_pos    = (col == 1).sum()
    n_neg    = n_tested - n_pos
    pct      = 100 * n_pos / n_tested if n_tested > 0 else 0
    pw       = n_neg / n_pos if n_pos > 0 else float('inf')
    print(f"{t:<20} {n_tested:<10} {n_pos:<10} {pct:<10.1f}% {pw:.1f}x")

Target               N tested   N toxic    % toxic    pos_weight
--------------------------------------------------------------
NR-AR                7265       309        4.3       % 22.5x
NR-AR-LBD            6758       237        3.5       % 27.5x
NR-AhR               6549       768        11.7      % 7.5x
NR-Aromatase         5821       300        5.2       % 18.4x
NR-ER                6193       793        12.8      % 6.8x
NR-ER-LBD            6955       350        5.0       % 18.9x
NR-PPAR-gamma        6450       186        2.9       % 33.7x
SR-ARE               5832       942        16.2      % 5.2x
SR-ATAD5             7072       264        3.7       % 25.8x
SR-HSE               6467       372        5.8       % 16.4x
SR-MMP               5810       918        15.8      % 5.3x
SR-p53               6774       423        6.2       % 15.0x


In [ ]:
# ── Preprocess: build label matrix and create SCAFFOLD-based splits ────────────
#
# Label encoding:
#   1.0  = toxic (positive)
#   0.0  = non-toxic (negative)
#  -1.0  = not tested (masked out of loss)
#
# WHY a scaffold split (and not a random one):
#   A random split lets structurally near-identical molecules (same Bemis–Murcko
#   scaffold, a substituent moved) land in BOTH train and test. The model is then
#   rewarded at test time for recognising near-twins it already saw — ROC-AUC comes
#   out optimistically high and says little about novel chemistry. Tox21/MoleculeNet
#   is benchmarked with scaffold splitting precisely to prevent this leakage, so a
#   scaffold split is the honest, leaderboard-comparable estimate of generalisation.
#
# TRADE-OFF we accept on purpose:
#   A pure scaffold split cannot also be stratified by label, so the per-split toxic
#   rate will drift (rare/singleton scaffolds are pushed into val/test by design —
#   that is the novel-chemistry stress test). We print the rates below so the shift
#   is visible and auditable rather than hidden.

from collections import defaultdict
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")   # silence RDKit parse chatter

df = df_raw.copy()
df = df[df[TASK_COLUMNS].notna().any(axis=1)]   # drop rows with no labels at all

label_matrix = df[TASK_COLUMNS].fillna(-1).values.astype(np.float32)
smiles_arr   = df[SMILES_COL].values

processed_df = pd.DataFrame({
    "smiles": smiles_arr,
    "labels": [row.tolist() for row in label_matrix],
}).reset_index(drop=True)

def murcko_scaffold(smiles: str) -> str:
    """Bemis–Murcko scaffold SMILES; unparseable/empty molecules group on their own."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return f"__unparseable__:{smiles}"
    scaff = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    return scaff if scaff else f"__empty__:{smiles}"

# Group molecule row-indices by their scaffold
scaffold_to_idx = defaultdict(list)
for idx, smi in enumerate(processed_df["smiles"]):
    scaffold_to_idx[murcko_scaffold(smi)].append(idx)

# Deterministic MoleculeNet-style assignment: biggest scaffold sets fill train first,
# so singletons (rare scaffolds) cascade into val/test. Tie-break on first index for
# full reproducibility across runs.
scaffold_sets = sorted(
    scaffold_to_idx.values(), key=lambda s: (len(s), -s[0]), reverse=True
)

n_total      = len(processed_df)
train_cutoff = 0.70 * n_total
val_cutoff   = 0.85 * n_total          # 70 / 15 / 15

train_idx, val_idx, test_idx = [], [], []
for group in scaffold_sets:
    if len(train_idx) + len(group) <= train_cutoff:
        train_idx.extend(group)
    elif len(train_idx) + len(val_idx) + len(group) <= val_cutoff:
        val_idx.extend(group)
    else:
        test_idx.extend(group)

train_df = processed_df.iloc[train_idx].reset_index(drop=True)
val_df   = processed_df.iloc[val_idx].reset_index(drop=True)
test_df  = processed_df.iloc[test_idx].reset_index(drop=True)

n_scaffolds  = len(scaffold_sets)
n_singletons = sum(1 for g in scaffold_sets if len(g) == 1)
print(f"Split strategy: Bemis–Murcko scaffold (deterministic, no label leakage)")
print(f"Unique scaffolds: {n_scaffolds}  |  singleton scaffolds: {n_singletons}")
print("-" * 60)
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    larr      = np.array(d["labels"].tolist())
    any_toxic = (larr.max(axis=1) > 0).mean()
    print(f"{name:<6}: {len(d):>5} compounds ({len(d)/n_total:5.1%}) | "
          f"{any_toxic:.1%} have \u2265 1 toxic label")


---
## Section 3: Tokenization

Tokenization is identical to Milestone 1 — ChemBERTa tokenises SMILES the same way  
regardless of whether we're doing single- or multi-target classification.  
The only change is the label format: instead of a scalar int, each sample now carries  
a list of 12 floats.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f"Vocabulary size: {tokenizer.vocab_size}")

def tokenize_batch(examples):
    return tokenizer(
        examples['smiles'],
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

# Declare explicit feature types so HuggingFace knows labels are a fixed-length
# float sequence — this ensures correct tensor shapes when set_format is called.
features = Features({
    'smiles': Value('string'),
    'labels': Sequence(Value('float32'), length=NUM_TARGETS),
})

def make_dataset(df):
    ds = HFDataset.from_dict(
        {'smiles': df['smiles'].tolist(), 'labels': df['labels'].tolist()},
        features=features,
    )
    ds = ds.map(tokenize_batch, batched=True)
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds

train_dataset = make_dataset(train_df)
val_dataset   = make_dataset(val_df)
test_dataset  = make_dataset(test_df)

print("Tokenized datasets:")
print(f"  Train: {train_dataset}")
sample = train_dataset[0]
print(f"\ninput_ids shape: {sample['input_ids'].shape}")
print(f"labels shape:    {sample['labels'].shape}  (one float per target)")
print(f"labels sample:   {sample['labels']}")

Vocabulary size: 591


Map: 100%|██████████| 1175/1175 [00:00<00:00, 31453.20 examples/s]

Tokenized datasets:
  Train: Dataset({
    features: ['smiles', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 5481
})

input_ids shape: torch.Size([128])
labels shape:    torch.Size([12])  (one float per target)
labels sample:   tensor([ 0., -1.,  0.,  0., -1.,  0., -1., -1.,  0., -1., -1.,  1.])


---
## Section 4: Model — ChemBERTa with a 12-output Head

The only change from Milestone 1 is `num_labels=12`.  
`AutoModelForSequenceClassification` replaces the regression head with  
`Linear(600 → 12)` — 12 independent logits, one per Tox21 target.  
We set `problem_type="multi_label_classification"` to document intent,  
though our custom trainer handles the loss directly.

In [9]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_TARGETS,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,
    trust_remote_code=True,
)

total = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters:  {total:,}")
print(f"Memory footprint:  {base_model.get_memory_footprint() / 1e6:.1f} MB")
print("\nTop-level modules:")
for name, module in base_model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {type(module).__name__:<35} {n:>10,} params")

[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `199`.
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 58777.92it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
regression.out_proj.weight | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
norm_mean                  | UNEXPECTED | 
norm_std                   | UNEXPECTED | 
regression.out_proj.bias   | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider traini

Total parameters:  3,432,060
Memory footprint:  13.7 MB

Top-level modules:
  roberta              RobertaModel                         3,279,600 params
  classifier           RobertaClassificationHead              152,460 params


In [10]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 226,188 || all params: 3,658,248 || trainable%: 6.1830


---
## Section 5: Training

### Two changes from Milestone 1

**1. Masked BCE loss instead of cross-entropy**

Cross-entropy works on a single label. For 12 independent binary predictions we need  
`BCEWithLogitsLoss`, applied element-wise:

```
loss = BCEWithLogitsLoss(logits, targets, pos_weight=pw) * mask
```

The `mask` zeros out any position where the label is −1 (untested), so those entries  
contribute no gradient in either direction.

**2. Per-target positive weights**

Each target has a different imbalance ratio (NR-PPAR-gamma is 32x imbalanced; SR-ARE is only 5x).  
We compute `pos_weight[i] = n_negative[i] / n_positive[i]` for each target independently,  
then pass the full weight vector to `BCEWithLogitsLoss`.

In [11]:
# ── Per-target positive weights ───────────────────────────────────────────────
labels_train = np.array(train_df['labels'].tolist())   # [N, 12]

pos_weights_list = []
print(f"{'Target':<20} {'N pos':<8} {'N neg':<8} {'pos_weight'}")
print("-" * 50)
for i, task in enumerate(TASK_COLUMNS):
    col     = labels_train[:, i]
    tested  = col[col != -1]
    n_pos   = (tested == 1).sum()
    n_neg   = (tested == 0).sum()
    pw      = n_neg / n_pos if n_pos > 0 else 1.0
    pos_weights_list.append(pw)
    print(f"{task:<20} {n_pos:<8} {n_neg:<8} {pw:.1f}x")

pos_weights_tensor = torch.tensor(pos_weights_list, dtype=torch.float32)
print(f"\nWeight tensor shape: {pos_weights_tensor.shape}")

Target               N pos    N neg    pos_weight
--------------------------------------------------
NR-AR                204      4872     23.9x
NR-AR-LBD            156      4551     29.2x
NR-AhR               538      4014     7.5x
NR-Aromatase         215      3864     18.0x
NR-ER                561      3772     6.7x
NR-ER-LBD            253      4609     18.2x
NR-PPAR-gamma        137      4363     31.8x
SR-ARE               683      3409     5.0x
SR-ATAD5             182      4757     26.1x
SR-HSE               270      4261     15.8x
SR-MMP               652      3426     5.3x
SR-p53               295      4436     15.0x

Weight tensor shape: torch.Size([12])


In [12]:
# ── Multi-target evaluation metrics ──────────────────────────────────────────
# Reports ROC-AUC per target and mean ROC-AUC (primary metric).
# Metric names use underscores (HuggingFace Trainer rejects hyphens in names).

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(
        torch.tensor(logits, dtype=torch.float32)
    ).numpy()   # [N, 12]

    aucs = {}
    for i, task in enumerate(TASK_COLUMNS):
        mask       = labels[:, i] != -1
        if mask.sum() < 2:
            continue
        task_labels = labels[mask, i]
        task_probs  = probs[mask, i]
        try:
            auc = roc_auc_score(task_labels, task_probs)
            key = "auc_" + task.replace("-", "_")
            aucs[key] = round(float(auc), 4)
        except ValueError:
            pass

    mean_auc = round(float(np.mean(list(aucs.values()))), 4) if aucs else 0.0
    return {"mean_roc_auc": mean_auc, **aucs}

In [13]:
# ── Training Arguments ────────────────────────────────────────────────────────
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
use_mps  = not torch.cuda.is_available() and torch.backends.mps.is_available()

if use_bf16:   precision = "bf16 (CUDA)"
elif use_fp16: precision = "fp16 (CUDA)"
elif use_mps:  precision = "fp32 (MPS / Apple Silicon)"
else:          precision = "fp32 (CPU)"
print(f"Precision/device: {precision}")

training_args = TrainingArguments(
    output_dir=RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_grad_norm=1.0,
    fp16=use_fp16,
    bf16=use_bf16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="mean_roc_auc",
    greater_is_better=True,
    logging_steps=LOG_STEPS,
    report_to="wandb",
    run_name=RUN_NAME,
    dataloader_pin_memory=False,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    hub_strategy="every_save",
    seed=SEED,
)

print(f"Steps per epoch: {math.ceil(len(train_dataset) / BATCH_SIZE)}")
print(f"Total steps:     {math.ceil(len(train_dataset) / BATCH_SIZE) * EPOCHS}")

Precision/device: fp32 (MPS / Apple Silicon)
Steps per epoch: 172
Total steps:     1720


In [14]:
# ── Multi-target Trainer with masked BCE loss ─────────────────────────────────
#
# Key differences from Milestone 1's WeightedLossTrainer:
#   - BCEWithLogitsLoss (not cross-entropy) for independent binary outputs
#   - mask zeros out untested entries (-1) per sample per target
#   - pos_weight is a vector of 12 values, one per target

class MultiTargetTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels").float()   # [batch, 12]
        outputs = model(**inputs)
        logits  = outputs.logits                 # [batch, 12]

        # Mask: 1 where tested, 0 where label == -1
        mask    = (labels != -1).float()
        targets = labels.clamp(min=0)            # -1 → 0 (will be zeroed by mask)

        loss_per_element = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=pos_weights_tensor.to(logits.device),
            reduction='none',
        )

        # Average only over tested (mask == 1) entries
        loss = (loss_per_element * mask).sum() / mask.sum().clamp(min=1)
        return (loss, outputs) if return_outputs else loss


trainer = MultiTargetTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(f"Training on   {len(train_dataset):,} samples")
print(f"Validating on {len(val_dataset):,} samples")
print(f"\nStarting training... \U0001f680")
train_result = trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Training on   5,481 samples
Validating on 1,175 samples

Starting training... 🚀


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Mean Roc Auc,Auc Nr Ar,Auc Nr Ar Lbd,Auc Nr Ahr,Auc Nr Aromatase,Auc Nr Er,Auc Nr Er Lbd,Auc Nr Ppar Gamma,Auc Sr Are,Auc Sr Atad5,Auc Sr Hse,Auc Sr Mmp,Auc Sr P53
50,1.415427,1.244936,0.718800,0.749800,0.812600,0.768400,0.806200,0.614700,0.731200,0.644200,0.647200,0.720200,0.653200,0.759700,0.717900
100,1.224117,1.181218,0.762100,0.767000,0.831300,0.818700,0.794700,0.698200,0.750900,0.758900,0.698500,0.778500,0.673100,0.823500,0.752400
150,1.189187,1.089395,0.748300,0.766500,0.831200,0.774200,0.794000,0.688800,0.736700,0.740800,0.686700,0.741000,0.676300,0.795400,0.748600
200,0.999085,1.060341,0.762700,0.765300,0.834400,0.805700,0.793300,0.707600,0.753500,0.762900,0.697300,0.756700,0.692400,0.818600,0.765000
250,1.048485,1.032142,0.774500,0.768000,0.837500,0.840000,0.788800,0.724800,0.769900,0.754300,0.717500,0.771800,0.701300,0.848700,0.771300
300,0.943377,1.013012,0.782000,0.769400,0.839900,0.847200,0.789500,0.727900,0.778400,0.756900,0.736500,0.782500,0.714800,0.861500,0.779400
350,0.895436,0.995204,0.791200,0.774100,0.848000,0.866700,0.792400,0.734500,0.784200,0.779500,0.743500,0.790000,0.720900,0.878000,0.782900
400,0.971214,1.002806,0.793500,0.775200,0.843400,0.874900,0.795600,0.738500,0.786500,0.784900,0.746800,0.789700,0.723200,0.879700,0.783100
450,0.982219,0.978769,0.800900,0.775900,0.849500,0.882200,0.809600,0.742100,0.786700,0.793000,0.751100,0.797300,0.744000,0.887900,0.791600
500,1.032031,0.974812,0.801100,0.775500,0.855000,0.885600,0.810800,0.741100,0.789900,0.787100,0.752100,0.793000,0.741800,0.889600,0.792000


---
## Section 6: Evaluation

We evaluate two ways:
1. **ROC-AUC per target** — measures ranking quality regardless of threshold
2. **Per-target threshold tuning** — finds the decision boundary that maximises F1 on the validation set, then reports recall/precision on the test set

In [15]:
# ── ROC-AUC per target on the test set ───────────────────────────────────────
test_preds  = trainer.predict(test_dataset)
test_logits = test_preds.predictions           # [N, 12]
test_labels = test_preds.label_ids             # [N, 12]
test_probs  = torch.sigmoid(
    torch.tensor(test_logits, dtype=torch.float32)
).numpy()

print(f"{'Target':<20} {'ROC-AUC':<10} {'N tested':<10} {'% toxic'}")
print("-" * 52)
aucs = {}
for i, task in enumerate(TASK_COLUMNS):
    mask        = test_labels[:, i] != -1
    task_labels = test_labels[mask, i]
    task_probs_ = test_probs[mask, i]
    n_tested    = mask.sum()
    pct_toxic   = 100 * task_labels.mean()
    try:
        auc = roc_auc_score(task_labels, task_probs_)
        aucs[task] = auc
        print(f"{task:<20} {auc:<10.4f} {n_tested:<10} {pct_toxic:.1f}%")
    except ValueError:
        print(f"{task:<20} {'N/A':<10} {n_tested:<10} {pct_toxic:.1f}%")

print(f"\n{'Mean ROC-AUC':<20} {np.mean(list(aucs.values())):.4f}")

Target               ROC-AUC    N tested   % toxic
----------------------------------------------------
NR-AR                0.8319     1094       4.2%
NR-AR-LBD            0.8316     1031       4.6%
NR-AhR               0.8624     996        11.6%
NR-Aromatase         0.8764     863        5.1%
NR-ER                0.7135     921        11.7%
NR-ER-LBD            0.7315     1044       4.3%
NR-PPAR-gamma        0.7594     966        2.6%
SR-ARE               0.7884     890        14.9%
SR-ATAD5             0.8393     1062       3.9%
SR-HSE               0.7831     970        5.9%
SR-MMP               0.8784     872        16.4%
SR-p53               0.8506     1018       6.5%

Mean ROC-AUC         0.8122


In [17]:
# ── Per-target threshold tuning on validation set ─────────────────────────────
# Search 0.05 → 0.95 in steps of 0.05. If the best threshold lands at the top
# of the range it means the true optimum is even higher — extend accordingly.

val_preds  = trainer.predict(val_dataset)
val_probs  = torch.sigmoid(
    torch.tensor(val_preds.predictions, dtype=torch.float32)
).numpy()
val_labels = val_preds.label_ids

best_thresholds = {}
THRESHOLD_RANGE = np.arange(0.05, 1.00, 0.05)

print(f"{'Target':<20} {'Threshold':<12} {'Val F1':<10} {'Recall':<10} {'Precision'}")
print("-" * 65)
for i, task in enumerate(TASK_COLUMNS):
    mask = val_labels[:, i] != -1
    tl   = val_labels[mask, i]
    tp   = val_probs[mask, i]

    best_t, best_f = 0.5, 0.0
    for t in THRESHOLD_RANGE:
        preds_t = (tp >= t).astype(int)
        f = f1_score(tl, preds_t, zero_division=0)
        if f > best_f:
            best_f, best_t = f, t

    # Warn if threshold is at the boundary of the search range
    ceiling = round(best_f, 3) == round(best_f, 3) and best_t >= THRESHOLD_RANGE[-1] - 0.01
    flag = " ⚠ at ceiling" if ceiling else ""

    best_thresholds[task] = round(float(best_t), 2)
    preds_best = (tp >= best_t).astype(int)
    from sklearn.metrics import recall_score, precision_score
    rec  = recall_score(tl, preds_best, zero_division=0)
    prec = precision_score(tl, preds_best, zero_division=0)
    print(f"{task:<20} {best_t:<12.2f} {best_f:<10.3f} {rec:<10.3f} {prec:.3f}{flag}")

print(f"\nThresholds saved to `best_thresholds` dict.")

Target               Threshold    Val F1     Recall     Precision
-----------------------------------------------------------------
NR-AR                0.95         0.465      0.339      0.741 ⚠ at ceiling
NR-AR-LBD            0.95         0.526      0.441      0.652 ⚠ at ceiling
NR-AhR               0.75         0.558      0.588      0.532
NR-Aromatase         0.85         0.272      0.268      0.275
NR-ER                0.70         0.441      0.435      0.446
NR-ER-LBD            0.85         0.397      0.462      0.348
NR-PPAR-gamma        0.85         0.400      0.458      0.355
SR-ARE               0.65         0.463      0.516      0.419
SR-ATAD5             0.80         0.339      0.463      0.268
SR-HSE               0.85         0.390      0.333      0.469
SR-MMP               0.85         0.613      0.553      0.687
SR-p53               0.85         0.309      0.274      0.354

Thresholds saved to `best_thresholds` dict.


---
### Confidence Calibration (Temperature Scaling)

ROC-AUC tells us the model *ranks* molecules well, but not whether a predicted
"0.92" actually means a 92% chance of activity. Networks trained with weighted
BCE on imbalanced data are typically **overconfident**. Temperature scaling fixes
this: we fit a single scalar **T** per endpoint on the validation logits and
divide the logits by T before the sigmoid.

- The model is **frozen** — this is a post-hoc rescaling, not retraining.
- T is fit by minimizing BCE (NLL) on the **validation** set; we report
  calibration error (**ECE**) on the held-out **test** set so the improvement is honest.
- Dividing logits by a positive constant is **monotonic**, so **ROC-AUC is
  provably unchanged** — we print before/after AUC per endpoint to prove it.
- **T > 1** ⇒ the model was overconfident (probabilities pulled toward 0.5);
  **T < 1** ⇒ underconfident (pushed toward the extremes).

**Running this on a fresh runtime (no retraining needed).** The cell below is
read-only with respect to the trained model. If you have NOT run the training
cells this session, it downloads the already-deployed model from the Hub into a
throwaway predict-only trainer and never touches your published weights. Just run,
in order: the install/imports cell, **Configuration** (Model & Data), the data
load, the scaffold split, and the tokenization cell — then this cell. You can
skip every model-build, LoRA, training, save, and push cell.

In [ ]:
# == Confidence calibration: per-endpoint temperature scaling =================
# Fit one temperature T per endpoint on the VALIDATION logits (model FROZEN),
# then report calibration error (ECE) on the HELD-OUT TEST set. Temperature
# scaling is a monotonic transform, so ROC-AUC is provably unchanged -- verified.
#
# READ-ONLY w.r.t. the trained model: this cell never calls .train(),
# .save_model(), or .push_to_hub(). On a fresh runtime (training cells not run)
# it loads the deployed model from the Hub into a THROWAWAY predict-only trainer
# (`_calib_trainer`) and never rebinds your `trainer`/`model`/`tokenizer`, so the
# published weights are untouched.
import json
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

DEPLOYED_MODEL_REPO = "mike-malloy/chemberta-tox21-multitarget-scaffold-20260603_1643"


def _get_predictor():
    """Prefer the in-memory trained `trainer`; otherwise build a read-only,
    predict-only Trainer around the deployed Hub model. Never clobbers globals."""
    try:
        trainer  # noqa: F821 -- defined only if you ran the training cells
        print("Using in-memory `trainer` (the model you just trained).")
        return trainer
    except NameError:
        pass

    print(f"No in-memory trainer -- loading deployed model read-only:\n  {DEPLOYED_MODEL_REPO}")
    from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
    from peft import PeftModel

    def _hf_token():
        try:
            from google.colab import userdata
            return userdata.get("HF_TOKEN")
        except Exception:
            import os
            return os.environ.get("HF_TOKEN")

    tok = _hf_token()
    if tok:
        from huggingface_hub import login
        login(tok)  # repo is private; harmless if already logged in

    # The deployed repo is a PEFT *adapter* repo (adapter_config.json +
    # adapter_model.safetensors), NOT a merged model. Loading it with a plain
    # AutoModelForSequenceClassification gives the base ChemBERTa with its default
    # 199-class head -> "Target size 12 vs input size 199" mismatch. Correct path:
    # build the base with our 12-label head, then overlay the trained adapter.
    # This mirrors notebook cells 15/16 and backend/inference.py exactly.
    _base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=NUM_TARGETS,
        problem_type="multi_label_classification",
        ignore_mismatched_sizes=True,
        trust_remote_code=True,
    )
    _base.config.id2label = {i: t for i, t in enumerate(TASK_COLUMNS)}
    _base.config.label2id = {t: i for i, t in enumerate(TASK_COLUMNS)}
    _model = PeftModel.from_pretrained(_base, DEPLOYED_MODEL_REPO)  # read-only overlay
    _model.eval()
    _args = TrainingArguments(
        output_dir="calib_tmp",
        per_device_eval_batch_size=64,
        report_to="none",
        push_to_hub=False,
        dataloader_pin_memory=False,
    )
    return Trainer(model=_model, args=_args)  # bound locally, not to `trainer`


def fit_temperature(logits, labels, max_iter=200):
    """Fit scalar T>0 minimizing BCE(logits / T, labels). Model stays frozen."""
    if labels.sum() == 0 or labels.sum() == len(labels):
        return 1.0  # one-class slice -- nothing to calibrate
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.float32)
    log_T = torch.zeros(1, requires_grad=True)            # T = exp(log_T) keeps T > 0
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=max_iter)
    bce = torch.nn.BCEWithLogitsLoss()

    def closure():
        opt.zero_grad()
        loss = bce(logits_t / torch.exp(log_T), labels_t)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(log_T).item())


def expected_calibration_error(probs, labels, n_bins=10):
    """Bin by predicted prob; sum |mean_prob - observed_freq| weighted by bin size."""
    edges = np.linspace(0, 1, n_bins + 1)
    N = len(probs)
    e = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        m = (probs >= lo) & (probs < hi) if b < n_bins - 1 else (probs >= lo) & (probs <= hi)
        if m.sum() == 0:
            continue
        e += (m.sum() / N) * abs(probs[m].mean() - labels[m].mean())
    return e


# -- Get val + test logits (in-memory trainer, or deployed model read-only) ----
_predictor   = _get_predictor()
val_preds_c  = _predictor.predict(val_dataset)
test_preds_c = _predictor.predict(test_dataset)
val_logits_all,  val_labels_all  = val_preds_c.predictions,  val_preds_c.label_ids
test_logits_all, test_labels_all = test_preds_c.predictions, test_preds_c.label_ids

temperatures = {}
ece_before_list, ece_after_list = [], []
print(f"{'Target':<16}{'T':>7}{'ECE_before':>13}{'ECE_after':>12}{'AUC_before':>13}{'AUC_after':>12}")
print("-" * 73)
for i, task in enumerate(TASK_COLUMNS):
    # fit T on the validation slice (masked)
    vmask = val_labels_all[:, i] != -1
    T = fit_temperature(val_logits_all[vmask, i], val_labels_all[vmask, i])
    temperatures[task] = round(T, 4)

    # evaluate calibration on the held-out TEST slice
    tmask = test_labels_all[:, i] != -1
    tl    = test_labels_all[tmask, i]
    tlog  = test_logits_all[tmask, i]
    p_before = 1.0 / (1.0 + np.exp(-tlog))
    p_after  = 1.0 / (1.0 + np.exp(-tlog / T))
    ece_b = expected_calibration_error(p_before, tl)
    ece_a = expected_calibration_error(p_after,  tl)
    try:
        auc_b, auc_a = roc_auc_score(tl, p_before), roc_auc_score(tl, p_after)
    except ValueError:
        auc_b = auc_a = float("nan")
    ece_before_list.append(ece_b)
    ece_after_list.append(ece_a)
    print(f"{task:<16}{T:>7.3f}{ece_b:>13.4f}{ece_a:>12.4f}{auc_b:>13.4f}{auc_a:>12.4f}")

print("-" * 73)
print(f"{'MEAN':<16}{'':>7}{np.mean(ece_before_list):>13.4f}{np.mean(ece_after_list):>12.4f}")
print("\nAUC_before == AUC_after on every row confirms temperature scaling preserves ranking.")

# -- Persist for the backend ---------------------------------------------------
with open("calibration_temperatures.json", "w") as f:
    json.dump(temperatures, f, indent=2)
print("\nSaved -> calibration_temperatures.json")
print("\nPaste into backend/inference.py:\n")
print("TEMPERATURES = {")
for task in TASK_COLUMNS:
    print(f"    {task!r}: {temperatures[task]},")
print("}")


## Applicability-domain reference export

Saves the **exact scaffold-split training molecules** plus a data-driven out-of-domain threshold, for the backend's `applicability_domain` guardrail (flags inputs unlike anything the model trained on, so the UI can warn instead of showing false confidence).

**READ-ONLY w.r.t. the model:** uses `train_df` only — never trains, saves, or pushes weights.

Run path on a fresh runtime: cells 5, 7, 9, 10, 11 (these define `train_df`), then this cell. Download the JSON and drop it into `backend/ad_reference_smiles.json`.

In [ ]:
# == Applicability-domain reference export ====================================
import json
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

DEPLOYED_MODEL_REPO = "mike-malloy/chemberta-tox21-multitarget-scaffold-20260603_1643"
FP_RADIUS, FP_BITS, K = 2, 2048, 5
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

# Morgan fingerprints for every (parseable) training molecule
fps, kept = [], []
for s in train_df["smiles"].tolist():
    m = Chem.MolFromSmiles(s)
    if m is None:
        continue
    fps.append(_gen.GetFingerprint(m))
    kept.append(s)

# Data-driven out-of-domain threshold: for each training molecule, its mean Tanimoto
# to its K nearest OTHER training molecules (leave-one-out); take the 5th percentile.
# A query whose mean-top-K similarity falls below this sits on the sparse edge of the
# training distribution -> out of domain.
loo = []
for i, fp in enumerate(fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, fps)
    sims[i] = -1.0                                   # exclude self
    loo.append(float(np.mean(sorted(sims, reverse=True)[:K])))
threshold = float(np.percentile(loo, 5))

out = {
    "smiles": kept,
    "threshold": round(threshold, 4),
    "params": {"fp_radius": FP_RADIUS, "fp_bits": FP_BITS, "k": K},
    "provenance": {
        "dataset": DATASET_NAME,
        "split": "Bemis-Murcko scaffold, deterministic 70/15/15 (this notebook)",
        "role": "applicability-domain reference = exact model training molecules",
        "model_repo": DEPLOYED_MODEL_REPO,
        "n_train": len(kept),
        "threshold_rule": "5th percentile of per-molecule leave-one-out mean-top-K Tanimoto",
    },
}
with open("ad_reference_smiles.json", "w") as f:
    json.dump(out, f)

print(f"Wrote ad_reference_smiles.json: {len(kept)} molecules, threshold={threshold:.4f}")
print("Sanity -- leave-one-out mean-top-K similarity distribution:")
for p in (1, 5, 25, 50, 75):
    print(f"  {p:>2}th pct: {np.percentile(loo, p):.3f}")

# Download from Colab, then drop into backend/ad_reference_smiles.json
try:
    from google.colab import files
    files.download("ad_reference_smiles.json")
except Exception:
    print("(not on Colab -- file is in the working directory)")


In [19]:
# ── Per-target test recall summary ────────────────────────────────────────────
from sklearn.metrics import recall_score, precision_score, confusion_matrix

print(f"{'Target':<20} {'Recall':<10} {'Precision':<12} {'TP':<6} {'FN':<6} {'FP'}")
print("-" * 62)
for i, task in enumerate(TASK_COLUMNS):
    mask      = test_labels[:, i] != -1
    tl        = test_labels[mask, i]
    tp        = test_probs[mask, i]
    preds     = (tp >= best_thresholds[task]).astype(int)

    rec  = recall_score(tl, preds, zero_division=0)
    prec = precision_score(tl, preds, zero_division=0)
    cm   = confusion_matrix(tl, preds, labels=[0, 1])
    fn   = cm[1, 0] if cm.shape == (2, 2) else 0
    tp_  = cm[1, 1] if cm.shape == (2, 2) else 0
    fp   = cm[0, 1] if cm.shape == (2, 2) else 0
    print(f"{task:<20} {rec:<10.3f} {prec:<12.3f} {tp_:<6} {fn:<6} {fp}")

Target               Recall     Precision    TP     FN     FP
--------------------------------------------------------------
NR-AR                0.543      0.781        25     21     7
NR-AR-LBD            0.489      0.793        23     24     6
NR-AhR               0.500      0.558        58     58     46
NR-Aromatase         0.386      0.395        17     27     26
NR-ER                0.361      0.382        39     69     63
NR-ER-LBD            0.400      0.265        18     27     50
NR-PPAR-gamma        0.160      0.174        4      21     19
SR-ARE               0.474      0.404        63     70     93
SR-ATAD5             0.293      0.218        12     29     43
SR-HSE               0.246      0.424        14     43     19
SR-MMP               0.406      0.682        58     85     27
SR-p53               0.167      0.306        11     55     25


---
## Section 7: Inference

`predict_toxicity` now returns a prediction for all 12 targets at once,  
using the per-target thresholds tuned on the validation set.

In [20]:
def predict_toxicity(smiles: str, thresholds: dict = None) -> dict:
    """
    Predict toxicity across all 12 Tox21 targets for a single SMILES string.

    Args:
        smiles:     A valid SMILES string.
        thresholds: Dict mapping target name → decision threshold.
                    Defaults to 0.5 for all targets; use `best_thresholds`
                    from the evaluation cell for calibrated predictions.

    Returns:
        Dict mapping target name → {probability, prediction}
    """
    if thresholds is None:
        thresholds = {t: 0.5 for t in TASK_COLUMNS}

    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits[0]).cpu().numpy()

    return {
        task: {
            "probability": round(float(probs[i]), 4),
            "prediction":  "TOXIC" if probs[i] >= thresholds[task] else "NON-TOXIC",
        }
        for i, task in enumerate(TASK_COLUMNS)
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
molecules = {
    "Aspirin":           "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":          "Cn1c(=O)c2c(ncn2C)n(c1=O)C",
    "Cisplatin (chemo)": "[Pt](Cl)(Cl)(N)N",
    "Benzene":           "c1ccccc1",
}

for mol_name, smi in molecules.items():
    results = predict_toxicity(smi, thresholds=best_thresholds)
    toxic_targets = [t for t, r in results.items() if r['prediction'] == 'TOXIC']
    print(f"\n{mol_name} ({smi})")
    print(f"  Flagged targets: {toxic_targets if toxic_targets else 'none'}")
    print(f"  {'Target':<20} {'P(toxic)':<12} {'Prediction'}")
    print(f"  {'-'*44}")
    for task, r in results.items():
        marker = " <--" if r['prediction'] == 'TOXIC' else ""
        print(f"  {task:<20} {r['probability']:<12.4f} {r['prediction']}{marker}")


Aspirin (CC(=O)Oc1ccccc1C(=O)O)
  Flagged targets: none
  Target               P(toxic)     Prediction
  --------------------------------------------
  NR-AR                0.1718       NON-TOXIC
  NR-AR-LBD            0.1278       NON-TOXIC
  NR-AhR               0.1256       NON-TOXIC
  NR-Aromatase         0.0196       NON-TOXIC
  NR-ER                0.2809       NON-TOXIC
  NR-ER-LBD            0.1852       NON-TOXIC
  NR-PPAR-gamma        0.3889       NON-TOXIC
  SR-ARE               0.1226       NON-TOXIC
  SR-ATAD5             0.1961       NON-TOXIC
  SR-HSE               0.1138       NON-TOXIC
  SR-MMP               0.0261       NON-TOXIC
  SR-p53               0.0718       NON-TOXIC

Caffeine (Cn1c(=O)c2c(ncn2C)n(c1=O)C)
  Flagged targets: none
  Target               P(toxic)     Prediction
  --------------------------------------------
  NR-AR                0.1615       NON-TOXIC
  NR-AR-LBD            0.1303       NON-TOXIC
  NR-AhR               0.3884       NON-TOXIC
  

---
## Section 8: Save, Push to Hub, and Log Metadata

In [21]:
trainer.save_model(RUN_NAME)
tokenizer.save_pretrained(RUN_NAME)
print(f"\u2705 Model saved locally to: {RUN_NAME}/")

trainer.push_to_hub(
    commit_message=f"Multi-target ChemBERTa-77M on all 12 Tox21 targets "
                   f"| mean ROC-AUC: {np.mean(list(aucs.values())):.4f}"
)
print(f"\u2705 Model pushed to: https://huggingface.co/{HUB_MODEL_NAME}")

metadata = {
    "experiment": {
        "run_name":    RUN_NAME,
        "timestamp":   datetime.now().isoformat(),
        "hub_model":   HUB_MODEL_NAME,
        "milestone":   2,
    },
    "data": {
        "dataset":        DATASET_NAME,
        "targets":        TASK_COLUMNS,
        "num_targets":    NUM_TARGETS,
        "train_samples":  len(train_dataset),
        "val_samples":    len(val_dataset),
        "test_samples":   len(test_dataset),
        "seed":           SEED,
        "split_strategy": "bemis_murcko_scaffold",
        "split_fractions": {"train": 0.70, "val": 0.15, "test": 0.15},
        "num_scaffolds":  int(n_scaffolds),
    },
    "model": {
        "base_model":     BASE_MODEL,
        "lora_r":         LORA_R,
        "lora_alpha":     LORA_ALPHA,
        "lora_dropout":   LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
    },
    "training": {
        "epochs":         EPOCHS,
        "batch_size":     BATCH_SIZE,
        "learning_rate":  LEARNING_RATE,
        "weight_decay":   WEIGHT_DECAY,
        "loss":           "masked_bce_with_per_target_pos_weights",
    },
    "results": {
        "per_target_auc":  {t: round(v, 4) for t, v in aucs.items()},
        "mean_roc_auc":    round(np.mean(list(aucs.values())), 4),
        "thresholds":      best_thresholds,
    },
}

with open(f"{RUN_NAME}/experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\n\u2705 Metadata saved")
print(json.dumps(metadata["results"], indent=2))

Processing Files (2 / 2): 100%|██████████|  912kB /  912kB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  


✅ Model saved locally to: chemberta-tox21-multitarget-20260519_0859/


Processing Files (2 / 2): 100%|██████████|  912kB /  912kB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  


✅ Model pushed to: https://huggingface.co/mike-malloy/chemberta-tox21-multitarget-20260519_0859

✅ Metadata saved
{
  "per_target_auc": {
    "NR-AR": 0.8319,
    "NR-AR-LBD": 0.8316,
    "NR-AhR": 0.8624,
    "NR-Aromatase": 0.8764,
    "NR-ER": 0.7135,
    "NR-ER-LBD": 0.7315,
    "NR-PPAR-gamma": 0.7594,
    "SR-ARE": 0.7884,
    "SR-ATAD5": 0.8393,
    "SR-HSE": 0.7831,
    "SR-MMP": 0.8784,
    "SR-p53": 0.8506
  },
  "mean_roc_auc": 0.8122,
  "thresholds": {
    "NR-AR": 0.95,
    "NR-AR-LBD": 0.95,
    "NR-AhR": 0.75,
    "NR-Aromatase": 0.85,
    "NR-ER": 0.7,
    "NR-ER-LBD": 0.85,
    "NR-PPAR-gamma": 0.85,
    "SR-ARE": 0.65,
    "SR-ATAD5": 0.8,
    "SR-HSE": 0.85,
    "SR-MMP": 0.85,
    "SR-p53": 0.85
  }
}


In [22]:
wandb.finish()
print("\u2705 Training complete!")
print(f"   Mean ROC-AUC:    {np.mean(list(aucs.values())):.4f}")
print(f"   WandB dashboard: https://wandb.ai/{PROJECT_NAME}")
print(f"   HuggingFace Hub: https://huggingface.co/{HUB_MODEL_NAME}")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/auc_NR_AR,▁▅▅▄▅▅▆▇▇▇▇███▇██▇▇█▇██▇███████████
eval/auc_NR_AR_LBD,▁▃▃▄▄▄▅▅▆▆▇▇▇▇▇▇▇▇▇████████████████
eval/auc_NR_AhR,▁▄▁▃▅▅▆▇▇▇▇▇▇██████████████████████
eval/auc_NR_Aromatase,▅▂▂▂▁▁▂▂▅▆▆▇█▇▇▇▇▇▇▇▇██████████████
eval/auc_NR_ER,▁▅▅▆▆▇▇▇▇▇▇▇▇██████████████████████
eval/auc_NR_ER_LBD,▁▃▂▃▅▆▇▇▇▇▇▇███████████████████████
eval/auc_NR_PPAR_gamma,▁▆▅▆▆▆▇▇████▇▇▇████████████████████
eval/auc_SR_ARE,▁▄▃▄▅▆▇▇▇▇▇▇▇▇█████████████████████
eval/auc_SR_ATAD5,▁▅▃▄▅▆▆▆▇▇▇▇▇██████████████████████
eval/auc_SR_HSE,▁▂▃▄▄▅▅▆▇▇▇▇▇██████████████████████
+29,...


✅ Training complete!
   Mean ROC-AUC:    0.8122
   WandB dashboard: https://wandb.ai/admet-toxicity
   HuggingFace Hub: https://huggingface.co/mike-malloy/chemberta-tox21-multitarget-20260519_0859


---
## What's Next?

With multi-target classification working, the natural next milestones are:

**Milestone 3 — Explainability:**  
Use attention weights or integrated gradients to highlight which atoms in a SMILES  
string drive each toxicity prediction. Turns a probability score into a chemical insight.

**Milestone 4 — Molecular similarity retrieval:**  
Build a FAISS index over training set embeddings. For any query molecule, retrieve  
the most structurally similar training examples — useful for explaining predictions by analogy.

**Milestone 5 — Streamlit UI:**  
A web app where users draw molecules or paste SMILES and see all 12 toxicity predictions  
with confidence scores, highlighted atoms, and similar known compounds.

---
*Project: ADMET Toxicity Prediction with ChemBERTa | Milestone 2*